# 🤖 VGA: Visual Geometry Alignment Policy for LIBERO Manipulation
### End-to-End Training & Closed-Loop MuJoCo Simulation Benchmark on Kaggle GPU

**Architecture Highlights**:
- **Vision Backbone**: Pretrained `google/siglip-base-patch16-256` (frozen, 93.3M params)
- **Language Backbone**: Pretrained `HuggingFaceTB/SmolLM2-135M` (frozen, 134.5M params)
- **VGA Policy Head**: RayRoPE Spatial Cross-Attention + Action DiT Expert (trainable, 36.3M params)
- **Total Model Capacity**: **263.7M parameters** (strictly under the 0.5B ceiling)
- **Real-Time Latency**: **~7.7 ms / chunk (129.1 Hz)** on Tesla T4 Tensor Cores (exceeds 50 Hz specification)

---

### Pipeline Steps:
1. **System & Environment Setup**: Headless MuJoCo OpenGL, LeRobot, and dependencies.
2. **Repository Sync**: Clone or update `vga-poc` to get the latest models and fixes.
3. **GPU Diagnostics**: Check accelerator availability (Tesla T4 x 2 or P100).
4. **VGA Policy Training**: 10-shot demonstration training (500 steps, ~3.5 min).
5. **Closed-Loop MuJoCo Evaluation**: Interactive simulation rollouts on LIBERO-Spatial Task 0.
6. **In-Notebook Video Playback**: Render MP4 rollout replays directly in the notebook.

## Step 1: Install System Libraries and Python Dependencies
Installs LeRobot, MuJoCo, OSMesa headless graphics, and manipulation packages.

In [ ]:
# 1. System packages for headless OpenGL/MuJoCo rendering in Kaggle Docker
!apt-get update -qq && apt-get install -y -qq libgl1-mesa-glx libosmesa6-dev

# 2. Python dependencies: av (video decoding), LeRobot, MuJoCo, and LIBERO simulation suite
!pip install --quiet "av<14" datasets num2words git+https://github.com/huggingface/lerobot.git zarr einops scipy torchvision peft accelerate matplotlib imageio imageio-ffmpeg "hf-libero>=0.1.4" mujoco

## Step 2: Clone or Update the `vga-poc` Repository
Safely clones `vga-poc` into `/kaggle/working/` or pulls the latest commit if already present.

In [ ]:
import os

repo_dir = "/kaggle/working/vga-poc"
if not os.path.exists(repo_dir):
    print("Cloning vga-poc repository...")
    !cd /kaggle/working && git clone https://github.com/Arkz-Deepak/vga-poc.git
else:
    print("Repository already exists. Pulling latest updates...")
    !cd /kaggle/working/vga-poc && git fetch origin && git reset --hard origin/main

## Step 3: Hardware Diagnostics & Mixed-Precision Check
Verifies CUDA availability and inspects available GPU accelerators.

In [ ]:
import os
import torch

# Configure headless MuJoCo environment
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")
if torch.cuda.is_available():
    num_gpus = torch.cuda.device_count()
    print(f"Detected {num_gpus} GPU(s):")
    for i in range(num_gpus):
        name = torch.cuda.get_device_name(i)
        mem = torch.cuda.get_device_properties(i).total_memory / (1024**3)
        print(f"  - GPU {i}: {name} ({mem:.2f} GB VRAM)")
else:
    print("⚠️ Running on CPU! Enable GPU under 'Settings' -> 'Accelerator' (choose T4 x 2 or P100).")

## Step 4: Train the VGA Policy (10-Shot Demonstration Benchmark)
Trains the VGA policy on 10 demonstration episodes of LIBERO-Spatial manipulation.
- **Backbones**: Frozen SigLIP + SmolLM2 (227.4M parameters).
- **Trainable Head**: Projector + RayRoPE + Action DiT Expert (36.3M parameters).
- **Duration**: 500 gradient steps takes ~3.5 minutes on 2x Tesla T4 GPUs.

In [ ]:
!python /kaggle/working/vga-poc/scripts/train_vga.py \
    --shots 10 \
    --steps 500 \
    --batch_size 8 \
    --output_dir /kaggle/working/vga-poc/checkpoints

## Step 5: Closed-Loop MuJoCo Simulation Evaluation
Runs live, closed-loop rollouts using the trained VGA checkpoint on **LIBERO-Spatial Task 0**:
`"pick up the black bowl between the plate and the ramekin and place it on the plate"`

**Evaluation Configuration**:
- **180° Camera Rotation (`flip_image=True`)**: Matches the official LeRobot `LiberoProcessorStep` camera convention.
- **Pure Policy Gripper Control**: Governed by the calibrated Schmitt trigger hysteresis (`schmitt_low=0.40, schmitt_high=0.60, min_hold_steps=60`) without arbitrary step cutoffs.
- **Video Recording**: High-fidelity 20 FPS MP4 rollouts saved to `/kaggle/working/vga-poc/results/videos/`.

In [ ]:
!python /kaggle/working/vga-poc/scripts/eval_mujoco_closed_loop.py \
    --checkpoint /kaggle/working/vga-poc/checkpoints/vga_libero_10shot.pt \
    --num_episodes 3 \
    --max_steps 280 \
    --video_dir /kaggle/working/vga-poc/results/videos \
    --output_json /kaggle/working/vga-poc/results/closed_loop_simulation_results.json

## Step 6: Interactive Rollout Video Playback
Embeds the generated simulation video directly in the notebook output.

In [ ]:
import glob
import os
from IPython.display import HTML, display
from base64 import b64encode

video_files = sorted(glob.glob("/kaggle/working/vga-poc/results/videos/*.mp4"))

if video_files:
    print(f"Found {len(video_files)} video replay(s):")
    for vf in video_files:
        print(f"  - {os.path.basename(vf)} ({os.path.getsize(vf) / 1024:.1f} KB)")
    
    # Display the most recent rollout video
    latest_video = video_files[-1]
    print(f"\nPlaying: {os.path.basename(latest_video)}")
    
    with open(latest_video, "rb") as f:
        mp4_bytes = f.read()
    data_url = "data:video/mp4;base64," + b64encode(mp4_bytes).decode()
    
    display(HTML(f"""
    <div style="text-align: center; margin: 15px 0;">
        <h4>Rollout Simulation Replay: {os.path.basename(latest_video)}</h4>
        <video width="640" height="640" controls autoplay loop style="border-radius: 8px; border: 1px solid #ddd;">
            <source src="{data_url}" type="video/mp4">
            Your browser does not support HTML5 video playback.
        </video>
    </div>
    """))
else:
    print("No MP4 simulation videos found in /kaggle/working/vga-poc/results/videos/")

## Step 7: View Metrics & Evaluation Summary
Displays the quantitative benchmark metrics saved in JSON format.

In [ ]:
import json
import os

results_path = "/kaggle/working/vga-poc/results/closed_loop_simulation_results.json"
if os.path.exists(results_path):
    with open(results_path, "r") as f:
        metrics = json.load(f)
    print(json.dumps(metrics, indent=2))
else:
    print(f"Results file not found at {results_path}")